In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mord import LogisticAT  # Ordinal logistic regression
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import tensorflow_addons as tfa
from keras import backend as K
from keras import losses
from imblearn.over_sampling import SMOTE, SMOTENC
from tensorflow.keras.layers import Lambda
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.sequence import pad_sequences


%config Completer.use_jedi = False

In [7]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [8]:
r = pd.read_csv("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/Blendshapes_Summary_1800_100.csv", index_col=0)
print(r.shape)
r.head()

(29708, 222)


,slice_index,filename,start_frame,end_frame,_neutral_mean,browDownLeft_mean,browDownRight_mean,browInnerUp_mean,browOuterUpLeft_mean,browOuterUpRight_mean,...,blinks_count,blinks_mean_duration,blinks_std_duration,blinks_min_duration,blinks_max_duration,yawns_count,yawns_mean_duration,yawns_std_duration,yawns_min_duration,yawns_max_duration
0,0,2_21_10_30fps_blendshape.csv,0,1799,0.0,0.020961,0.011672,0.091767,0.175344,0.169944,...,38,5.394737,2.604979,1,14,0,0.0,0.0,0,0
1,1,2_21_10_30fps_blendshape.csv,100,1899,0.0,0.023722,0.013317,0.084261,0.166261,0.161839,...,38,5.026316,2.259955,1,14,0,0.0,0.0,0,0
2,2,2_21_10_30fps_blendshape.csv,200,1999,0.0,0.024122,0.013600,0.087417,0.166378,0.161378,...,39,4.923077,2.309985,1,14,0,0.0,0.0,0,0
3,3,2_21_10_30fps_blendshape.csv,300,2099,0.0,0.024967,0.013989,0.085200,0.162722,0.157400,...,37,4.810811,2.132263,1,14,0,0.0,0.0,0,0
4,4,2_21_10_30fps_blendshape.csv,400,2199,0.0,0.026233,0.014633,0.082817,0.159100,0.153994,...,39,4.769231,2.070616,1,14,0,0.0,0.0,0,0


In [10]:
r['target'] = r['filename'].apply(lambda x:int(x.split('_')[2].replace('10','2').replace('0','0').replace('5','1')))
r['fold'] = r['filename'].apply(lambda x: int(x.split('_')[0]))
r['subject_id'] = r['filename'].apply(lambda x: int(x.split('_')[1]))
r['target'].value_counts()

2    10416
1     9782
0     9510
Name: target, dtype: int64

In [12]:
def step(val, steps, outputs):
    for i in range(len(steps)-1):
        low = steps[i]
        high = steps[i+1]
        if val >= low and val <= high:
            return outputs[i]
        
def report(model, x, y, output_dict):
    y_pred = [step(v,[0,0.33,0.66,1],[0,1,2]) for v in model.predict(x, verbose=0)]
#     y_pred = [step(v,[-1.5,-0.5,0.5,1.5],[-1,0,1]) for v in model.predict(x, verbose=0)]
#     y_pred = [step(v,[-3,-1,1,3],[-2,0,2]) for v in model.predict(x, verbose=0)]
    if output_dict:
        cr = classification_report(list(y), y_pred, output_dict=True)
        cr['confusion_matrix'] = list(confusion_matrix(list(y), y_pred))
        return cr
    
    print(classification_report(list(y), y_pred))
    print(confusion_matrix(list(y), y_pred))

In [13]:
def reshape_data_rnn(df, feature_columns=[], groupby_column='filename', max_slices=30, sortby_column='slice_index', target_column='target'):
    X, y = [], []
    grouped = df.groupby(groupby_column)
    
    for video, group in grouped:
        # Sort by slice_index to maintain temporal order
        group = group.sort_values(by=sortby_column)

        # Extract features
        slices = group[feature_columns].values.tolist()

        # Split into chunks of max_slices
        for i in range(0, len(slices), max_slices):
            chunk = slices[i:i + max_slices]
            X.append(chunk)
            y.append(group[target_column].iloc[0])  # Same label for all chunks from the same video

    # Pad sequences to the max_slices length
    X_padded = pad_sequences(X, maxlen=max_slices, padding='post', dtype='float32')

    # Convert labels to numpy array
    y = np.array(y)
    
    return X_padded, y

In [14]:
# First, define the train_single_fold function outside of the class
def _train_single_fold(args):
    """Helper function for parallel processing in test_model.
    
    Args:
        args: Tuple containing (model_instance, test_idx, val_idx, data)
    
    Returns:
        Dictionary containing model, history, and reports
    """
    self, i, j, data = args
    local_results = {}
    
    print(f"Starting Test {i} & Validation {j}")
    
    train = data[~data[self.cv_on].isin([i, j])].reset_index(drop=True)
    val = data[data[self.cv_on] == j].reset_index(drop=True)
    test = data[data[self.cv_on] == i].reset_index(drop=True)
    
    # List of columns to exclude from scaling
    exclude_cols = ['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id']
    feature_columns = data.columns.difference(exclude_cols)
    
    # Fit the scaler on the training data
    scaler = StandardScaler().fit(train[feature_columns])
    
    # Transform the training, validation, and test data
    train_scaled = scaler.transform(train[feature_columns])
    val_scaled = scaler.transform(val[feature_columns])
    test_scaled = scaler.transform(test[feature_columns])
    
    # Convert scaled arrays back to DataFrames and add the excluded columns
    train_scaled_df = pd.DataFrame(train_scaled, columns=feature_columns, index=train.index)
    val_scaled_df = pd.DataFrame(val_scaled, columns=feature_columns, index=val.index)
    test_scaled_df = pd.DataFrame(test_scaled, columns=feature_columns, index=test.index)
    
    # Concatenate the scaled features with the excluded columns
    train = pd.concat([train_scaled_df, train[exclude_cols]], axis=1)
    val = pd.concat([val_scaled_df, val[exclude_cols]], axis=1)
    test = pd.concat([test_scaled_df, test[exclude_cols]], axis=1)
    
    X_train, y_train = reshape_data_rnn(train, feature_columns, groupby_column='filename', max_slices=30, sortby_column='slice_index', target_column='target')
    X_val, y_val = reshape_data_rnn(val, feature_columns, groupby_column='filename', max_slices=30, sortby_column='slice_index', target_column='target')
    X_test, y_test = reshape_data_rnn(test, feature_columns, groupby_column='filename', max_slices=30, sortby_column='slice_index', target_column='target')
    
    classes = [0, 1, 2]
    class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight_dict = {}
    for c, w in zip(classes, class_weights):
        class_weight_dict[c] = w
    
    # Boost the weight for the middle class
    class_weight_dict[1] *= 1.5
    
    model = self.build_model((X_train.shape[1], X_train.shape[2]))
    
    # Train the model
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=self.epochs,
        batch_size=self.batch_size,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=self.patience, restore_best_weights=True)],
        verbose=0,  # Reduced verbosity for parallel execution
        class_weight=class_weight_dict
    )
    
    train_report = report(model, X_train, y_train, True)
    val_report = report(model, X_val, y_val, True)
    test_report = report(model, X_test, y_test, True)
    
    # Store results to return
    local_results = {
        'test': i,
        'val': j,
        'model': model,
        'history': history,
        'train_report': train_report,
        'val_report': val_report,
        'test_report': test_report
    }
    
    print(f"Completed Test {i} & Validation {j}")
    return local_results

In [25]:
class Model:
    def __init__(self, units, dropout_ratio, kernel_regularizer_l2, kernel_regularizer_l1, batch_size, num_classes, learning_rate, weight_decay, beta_1, beta_2, epsilon, cv_on, epochs, patience):
        self.units = units
        self.dropout_ratio = dropout_ratio
        self.kernel_regularizer_l2 = kernel_regularizer_l2
        self.kernel_regularizer_l1 = kernel_regularizer_l1
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.cv_on = cv_on #subject_id, fold
        self.epochs = epochs
        self.patience = patience
        self.models = []
        self.histories = []
        self.reports = []
        
        
    def build_model(self, input_shape):
        model = keras.Sequential()
        model.add(layers.Input(shape=(input_shape[0], input_shape[1])))  # (30, 218)

        # Add LSTM layers
        for u in self.units:
            model.add(layers.LSTM(u, activation='tanh', return_sequences=True, 
                                  kernel_regularizer=tf.keras.regularizers.l2(self.kernel_regularizer_l2)))
            model.add(layers.BatchNormalization())
            model.add(layers.Dropout(self.dropout_ratio))

        # Final LSTM layer without return_sequences
        model.add(layers.LSTM(self.units[-1], activation='tanh', 
                              kernel_regularizer=tf.keras.regularizers.l2(self.kernel_regularizer_l2)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(self.dropout_ratio))

        # Final output layer
        model.add(layers.Dense(1, activation='sigmoid'))

        model.compile(
            optimizer = keras.optimizers.AdamW(
                learning_rate=self.learning_rate,
                weight_decay=self.weight_decay,
                beta_1=self.beta_1,
                beta_2=self.beta_2,
                epsilon=self.epsilon
            ),
            loss='mse',
            metrics=['mae']
        )

        return model

    
    
    
    def get_reports(self):
        records = []

        for item in self.reports:
            base_info = {'test': item['test'], 'val': item['val']}
            for report_type in ['train_report', 'val_report', 'test_report']:
                report = item.get(report_type, {})
                for category, metrics in report.items():
                    if category == 'confusion_matrix':
                        cm_flat = np.array(metrics).flatten()
                        for i, value in enumerate(cm_flat):
                            records.append({**base_info, 'report_type': report_type, 'category': f'conf_matrix_{i}', 'metric': value})
                    elif isinstance(metrics, dict):
                        for metric, value in metrics.items():
                            records.append({**base_info, 'report_type': report_type, 'category': category, 'metric_name': metric, 'metric_value': value})
                    else:
                        records.append({**base_info, 'report_type': report_type, 'category': 'macro avg', 'metric_name': category, 'metric_value': metrics})

        return pd.DataFrame(records)

        
    
    # Now modify the test_model method
    def test_model(self, data):
        import concurrent.futures
        from tqdm import tqdm

        unique = data[self.cv_on].unique()
        test_val_pairs = []

        # Generate all valid test-validation pairs
        for i in unique:
            for j in unique:
                if i == j:
                    continue
                if i < 2 or j < 4:  # Note: I fixed this condition back to your original logic
                    continue
                test_val_pairs.append((i, j))
       
        print(f"Found {len(test_val_pairs)} valid test-validation pairs.")

        # Prepare arguments for parallel processing
        args_list = [(self, i, j, data) for i, j in test_val_pairs]

        # Run training in parallel
        all_results = []
        for args in args_list[:1]:
            all_results.append(_train_single_fold(args))
#         with concurrent.futures.ProcessPoolExecutor(max_workers=min(len(test_val_pairs), 4)) as executor:
#             futures = [executor.submit(_train_single_fold, args) for args in args_list]
#             for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Training folds"):
#                 all_results.append(future.result())

        # Process results
        for result in all_results:
            self.models.append(result['model'])
            self.histories.append(result['history'])
            self.reports.append({
                'test': result['test'],
                'val': result['val'],
                'train_report': result['train_report'],
                'val_report': result['val_report'],
                'test_report': result['test_report']
            })

        print(f"Completed all {len(test_val_pairs)} test-validation pairs.")

In [26]:
m = Model(units = [30,20,10], 
          dropout_ratio = 0.5, 
          kernel_regularizer_l2 = 0, 
          kernel_regularizer_l1 = 0, 
          batch_size = 16, 
          num_classes = 3, 
          learning_rate = 1e-3, 
          weight_decay = 1e-4, 
          beta_1 = 0.9, 
          beta_2 = 0.999, 
          epsilon = 1e-7, 
          cv_on = 'subject_id', 
          epochs = 1, 
          patience = 100)

m.test_model(r)

Found 3306 valid test-validation pairs.
Starting Test 21 & Validation 4
Completed Test 21 & Validation 4
Completed all 3306 test-validation pairs.


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

In [32]:
m.histories[0].history['val_loss']

[0.8050531148910522]

In [157]:
reports = m.get_reports()
reports[(reports['metric_name'] == 'accuracy') & (reports['report_type'] == 'train_report')]

,test,val,report_type,category,metric_name,metric_value,metric
12,21,4,train_report,macro avg,accuracy,0.604227,NaN
102,21,7,train_report,macro avg,accuracy,0.646494,NaN
192,21,1,train_report,macro avg,accuracy,0.639769,NaN
282,21,5,train_report,macro avg,accuracy,0.621518,NaN
372,21,20,train_report,macro avg,accuracy,0.623077,NaN
462,21,39,train_report,macro avg,accuracy,0.606495,NaN
543,21,50,train_report,macro avg,accuracy,0.592878,NaN
633,21,59,train_report,macro avg,accuracy,0.613833,NaN
723,21,57,train_report,macro avg,accuracy,0.626321,NaN
813,21,6,train_report,macro avg,accuracy,0.622010,NaN


In [146]:
for t, v in reports.groupby('test'):
    for r, d in v.groupby('val'):
        print(t, r)

21 1
21 4
21 5
21 6
21 7
21 20
21 39
21 50
21 57
21 59


In [160]:
reports[(reports['category'].isin(["conf_matrix_0", "conf_matrix_1", "conf_matrix_2", "conf_matrix_3", "conf_matrix_4", "conf_matrix_5", "conf_matrix_6", "conf_matrix_7", "conf_matrix_8"])) & 
        (reports['test']==21) & 
        (reports['val']==4) & 
        (reports['report_type']=='train_report')]

,test,val,report_type,category,metric_name,metric_value,metric
21,21,4,train_report,conf_matrix_0,NaN,NaN,265.0
22,21,4,train_report,conf_matrix_1,NaN,NaN,0.0
23,21,4,train_report,conf_matrix_2,NaN,NaN,67.0
24,21,4,train_report,conf_matrix_3,NaN,NaN,10.0
25,21,4,train_report,conf_matrix_4,NaN,NaN,0.0
26,21,4,train_report,conf_matrix_5,NaN,NaN,333.0
27,21,4,train_report,conf_matrix_6,NaN,NaN,2.0
28,21,4,train_report,conf_matrix_7,NaN,NaN,0.0
29,21,4,train_report,conf_matrix_8,NaN,NaN,364.0
